# 00.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/00-big-picture/05-whole-course-in-one-notebook/solutions.ipynb)

The setup cell below repeats the lesson's loading and cleaning so this notebook runs on its own.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, time
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

t0 = time.time()
raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(f"loaded {len(raw):,} rows in {time.time() - t0:.0f}s")

df = raw.dropna(subset=["Customer ID"]).copy()
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
df = df[~df["Invoice"].astype(str).str.startswith("C")]
df["amount"] = df["Quantity"] * df["Price"]
df["Customer ID"] = df["Customer ID"].astype(int)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def build(cutoff):
    cutoff = pd.Timestamp(cutoff)
    before, after = df[df["InvoiceDate"] < cutoff], df[df["InvoiceDate"] >= cutoff]
    g = before.groupby("Customer ID")
    c = pd.DataFrame({"recency_days": (cutoff - g["InvoiceDate"].max()).dt.days,
                      "frequency": g["Invoice"].nunique(), "monetary": g["amount"].sum(),
                      "avg_basket": g["amount"].sum() / g["Invoice"].nunique(),
                      "n_products": g["StockCode"].nunique(),
                      "tenure_days": (cutoff - g["InvoiceDate"].min()).dt.days,
                      "is_uk": (g["Country"].first() == "United Kingdom").astype(int)})
    c["churned"] = (~c.index.isin(after["Customer ID"].unique())).astype(int)
    return c

features = ["recency_days", "frequency", "monetary", "avg_basket", "n_products", "tenure_days", "is_uk"]

loaded 541,910 rows in 13s


## Exercise 1

In [2]:
for cutoff in ["2011-06-09", "2011-09-09"]:
    c = build(cutoff)
    Xtr, Xte, ytr, yte = train_test_split(c[features], c["churned"], test_size=0.25, random_state=42, stratify=c["churned"])
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    print(f"cut-off {cutoff}: {len(c):,} customers, churn rate {c['churned'].mean():.1%}, "
          f"forest accuracy {accuracy_score(yte, rf.predict(Xte)):.1%}")

cut-off 2011-06-09: 2,787 customers, churn rate 30.3%, forest accuracy 69.6%


cut-off 2011-09-09: 3,362 customers, churn rate 42.9%, forest accuracy 66.2%


An earlier cut-off has fewer customers in the "before" window (fewer had made a first purchase yet) and a different churn rate, because the 90 days after June are the quieter summer months. The trade-off in practice: an older cut-off gives you a label you can verify sooner, but the behaviour it learns from is further from today's.

## Exercise 2

In [3]:
c = build("2011-09-09")
Xtr, Xte, ytr, yte = train_test_split(c[features], c["churned"], test_size=0.25, random_state=42, stratify=c["churned"])
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print(f"forest: {accuracy_score(yte, rf.predict(Xte)):.1%}")
for days in [30, 60, 90, 120]:
    print(f"baseline recency > {days:>3}: {accuracy_score(yte, (Xte['recency_days'] > days).astype(int)):.1%}")

forest: 66.2%
baseline recency >  30: 57.7%
baseline recency >  60: 61.5%
baseline recency >  90: 62.9%
baseline recency > 120: 65.2%


The best single-threshold baseline gets close to the forest. Recency carries most of the signal; the forest adds a few points by combining it with frequency and spend. Whether those points are worth a model is exactly the kind of question framing (Step 1) should answer.

## Exercise 3

In [4]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
rfm_scaled = StandardScaler().fit_transform(np.log1p(c[["recency_days", "frequency", "monetary"]]))
c["segment"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit(rfm_scaled).labels_
c.groupby("segment").agg(customers=("churned", "size"), recency_days=("recency_days", "median"),
                         orders=("frequency", "median"), spent=("monetary", "median"),
                         churn_rate=("churned", "mean")).round(2)

,customers,recency_days,orders,spent,churn_rate
segment,,,,,
0,1008,64.5,3.0,1100.86,0.32
1,1460,152.0,1.0,253.54,0.62
2,447,13.0,2.0,541.51,0.40
3,447,9.0,9.0,3337.70,0.06


Typical names, matched by reading the medians: **champions** (recent, frequent, high spend), **regulars**, **new or occasional** (few orders, modest spend), **lapsed** (high recency, high churn rate). Your labels may differ; what matters is that each is a sentence marketing could act on.